# TCC — Aplicação de modelos de machine learning na previsão do CDI
## Seção 4.2 — Treinamento do modelo e seleção de hiperparâmetros

**Autor:** Marcos Paulo Galli dos Santos
**Orientador:** Prof. José Agostinho Baitello
**Centro Universitário FEI** — Bacharelado em Engenharia de Produção (2026)

Este notebook implementa o pipeline descrito nas seções 3.4 a 3.9 do TCC:

- **Alvo:** CDI médio realizado nos 22 dias úteis seguintes (anualizado, base 252 du)
- **Treino:** 02/01/2023 a 28/11/2025 — **Embargo:** 22 du — **Teste:** 02/01/2026 a 31/03/2026
- **Algoritmo:** XGBoost
- **Validação cruzada temporal** com `gap=22` entre treino e validação em cada fold
- **Métrica de seleção:** RMSE médio entre folds (penaliza erros grandes em pontos de inflexão)

A seção 4.2 não usa o período de teste — ele é reservado para 4.3 (comparação com a curva DI).


## 0. Setup do ambiente

In [ ]:
# Instala XGBoost se ainda não estiver disponível (Colab já vem com pandas, sklearn, etc.)
import subprocess, sys
print("xgboost", xgboost.__version__)


In [ ]:
# Upload dos arquivos no Colab.
# (e, se quiser, dados/dados_modelo_tcc_di_futuro.xlsx — só será usado em 4.3)


In [ ]:
import os, time, warnings
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
import xgboost as xgb

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)


In [ ]:
# Parâmetros do experimento
PATH_CDI = "dados/dados_modelo_tcc_cdi.xlsx"

DATA_TREINO_INI   = pd.Timestamp("2023-01-02")
DATA_TREINO_FIM   = pd.Timestamp("2025-11-28")
DATA_EMBARGO_INI  = pd.Timestamp("2025-12-01")
DATA_EMBARGO_FIM  = pd.Timestamp("2026-01-01")
DATA_TESTE_INI    = pd.Timestamp("2026-01-02")
DATA_TESTE_FIM    = pd.Timestamp("2026-03-31")

H = 22  # horizonte de previsão (dias úteis)


## 4.2.1 — Construção da base de modelagem

Carrega a base consolidada, constrói o alvo `y_t = média do CDI nos próximos 22 dias úteis`,
separa treino/embargo/teste e audita integridade (NaN, Copoms, perda por lag/embargo).


In [ ]:
# Carrega a base consolidada já tratada conforme 3.5
df = pd.read_excel(PATH_CDI, sheet_name="Diario_Consolidado")
df["Data"] = pd.to_datetime(df["Data"])
df = df.sort_values("Data").reset_index(drop=True)

# Calendário Copom (para auditoria)
copom = pd.read_excel(PATH_CDI, sheet_name="Calendario_Copom")
copom["Reunioes_Copom"] = pd.to_datetime(copom["Reunioes_Copom"])

print(f"Base consolidada: {df.shape[0]} linhas x {df.shape[1]} colunas")
print(f"Período: {df['Data'].min().date()} a {df['Data'].max().date()}")


In [ ]:
# Construção do alvo y_t = média de CDI nos dias [t+1, t+22]
df["y_22d"] = (
    df["CDI"].shift(-1).rolling(window=H, min_periods=H).mean().shift(-(H - 1))
)

# Conferência independente por loop explícito (mesmo cálculo, código distinto)
y_check = np.full(len(df), np.nan)
cdi_arr = df["CDI"].to_numpy()
for i in range(len(df) - H):
    y_check[i] = cdi_arr[i + 1 : i + 1 + H].mean()

mask_both = df["y_22d"].notna()
max_diff = np.abs(df.loc[mask_both, "y_22d"].values - y_check[mask_both.values]).max()
assert max_diff < 1e-10, "Construção do alvo divergiu da referência."
print(f"Construção do alvo OK (diferença entre métodos: {max_diff:.2e})")


In [ ]:
# Exemplo concreto da construção do alvo — pegamos uma data dentro do ciclo de cortes
idx_ex = df.index[df["Data"] == pd.Timestamp("2024-08-01")][0]
data_ex = df.loc[idx_ex, "Data"]
y_ex = df.loc[idx_ex, "y_22d"]
proximos = df.iloc[idx_ex + 1 : idx_ex + 1 + H][["Data", "CDI"]].copy()
proximos["Data"] = proximos["Data"].dt.date

print(f"Exemplo: t = {data_ex.date()}  |  CDI(t) = {df.loc[idx_ex,'CDI']}%  |  y_t = {y_ex:.4f}% a.a.")
print(f"Composto pela média dos 22 valores de CDI nos dias:")
print(proximos.to_string(index=False))
print(f"\nMédia conferida: {proximos['CDI'].mean():.4f}% a.a.")


In [ ]:
# Separação treino / embargo / teste
mask_treino  = (df["Data"] >= DATA_TREINO_INI)  & (df["Data"] <= DATA_TREINO_FIM)
mask_embargo = (df["Data"] >= DATA_EMBARGO_INI) & (df["Data"] <= DATA_EMBARGO_FIM)
mask_teste   = (df["Data"] >= DATA_TESTE_INI)   & (df["Data"] <= DATA_TESTE_FIM)

# Conjunto de features: todas as colunas exceto Data e o alvo
cols_drop = ["Data", "y_22d"]
feature_cols = [c for c in df.columns if c not in cols_drop]

df_treino = df.loc[mask_treino].dropna(subset=["y_22d"]).reset_index(drop=True)
X_treino = df_treino[feature_cols]
y_treino = df_treino["y_22d"]
dates_treino = df_treino["Data"]

print(f"Treino:  {len(df_treino)} obs ({dates_treino.min().date()} a {dates_treino.max().date()})")
print(f"Embargo: {mask_embargo.sum()} obs ({DATA_EMBARGO_INI.date()} a {DATA_EMBARGO_FIM.date()})")
print(f"Teste:   {mask_teste.sum()} obs ({DATA_TESTE_INI.date()} a {DATA_TESTE_FIM.date()})")
print(f"Número de features: {len(feature_cols)}")


In [ ]:
# Auditoria de integridade
warmup_ini = pd.Timestamp("2022-09-01")
warmup = df[(df["Data"] >= warmup_ini) & (df["Data"] < DATA_TREINO_INI)]
n_sem_alvo = int(df["y_22d"].isna().sum())

print("Observações por etapa:")
print(f"  Aquecimento (apenas para lags): {len(warmup)} obs")
print(f"  Treino efetivo:                 {len(df_treino)} obs")
print(f"  Embargo (22 du):                {mask_embargo.sum()} obs")
print(f"  Teste:                          {mask_teste.sum()} obs")
print(f"  Sem alvo (últimos 22 du):       {n_sem_alvo} obs")
print(f"  Total na base:                  {len(df)} obs")

# NaN nas features dentro do treino
nan_summary = X_treino.isna().sum()
nan_summary = nan_summary[nan_summary > 0].sort_values(ascending=False)
print(f"\nFeatures com NaN no treino: {len(nan_summary)}")
if len(nan_summary):
    print(nan_summary.to_string())
    print("Observação: XGBoost trata valores ausentes nativamente (default direction nos splits),")
    print("portanto não realizamos imputação. Os NaN remanescentes vêm do início da amostra,")
    print("onde variáveis derivadas (revisões Focus) ainda não dispõem de referência anterior.")


In [ ]:
# Distribuição das reuniões do Copom no período de treino
copom_treino = copom[(copom["Reunioes_Copom"] >= DATA_TREINO_INI) &
                     (copom["Reunioes_Copom"] <= DATA_TREINO_FIM)]
print(f"Reuniões Copom no período de treino: {len(copom_treino)}")
print(copom_treino["Reunioes_Copom"].dt.year.value_counts().sort_index().to_string())


## 4.2.2 — Validação cruzada temporal e busca em grade

Particionamento expansivo do treino em 5 folds sequenciais, com embargo de 22 du entre
o fim do treino do fold e o início da janela de validação (conforme 3.8 do TCC).
Métrica de seleção: RMSE médio entre folds.


In [ ]:
N_SPLITS = 5
GAP = H  # 22 du
tscv = TimeSeriesSplit(n_splits=N_SPLITS, gap=GAP)

folds_info = []
for i, (tr_idx, va_idx) in enumerate(tscv.split(X_treino)):
    folds_info.append({
        "Fold": i + 1,
        "Treino_inicio": dates_treino.iloc[tr_idx[0]].date(),
        "Treino_fim":    dates_treino.iloc[tr_idx[-1]].date(),
        "Treino_obs":    len(tr_idx),
        "Embargo_du":    GAP,
        "Val_inicio":    dates_treino.iloc[va_idx[0]].date(),
        "Val_fim":       dates_treino.iloc[va_idx[-1]].date(),
        "Val_obs":       len(va_idx),
    })
folds_df = pd.DataFrame(folds_info)
folds_df


In [ ]:
# Grade de hiperparâmetros — abrange os parâmetros listados em 3.8
param_grid = {
    "learning_rate":    [0.03, 0.05, 0.10],
    "n_estimators":     [400, 800],
    "max_depth":        [3, 5, 7],
    "subsample":        [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5],
    "reg_lambda":       [1.0, 2.0],
    "reg_alpha":        [0.0, 0.1],
}
param_names = list(param_grid.keys())
combos = list(product(*[param_grid[k] for k in param_names]))

print("Grade de hiperparâmetros:")
for k, v in param_grid.items():
    print(f"  {k:18s}: {v}")
print(f"\nTotal de combinações: {len(combos)}")
print(f"Total de fits (combinações x folds): {len(combos) * N_SPLITS}")


In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

base_params = {
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": 0,
}

print(f"Iniciando busca em grade ({len(combos)} combinações x {N_SPLITS} folds)...")
t0 = time.time()

resultados = []
for j, combo in enumerate(combos):
    params = {**base_params, **dict(zip(param_names, combo))}
    fold_rmses = []
    for tr_idx, va_idx in tscv.split(X_treino):
        model = xgb.XGBRegressor(**params)
        model.fit(X_treino.iloc[tr_idx], y_treino.iloc[tr_idx], verbose=False)
        pred = model.predict(X_treino.iloc[va_idx])
        fold_rmses.append(rmse(y_treino.iloc[va_idx], pred))
    resultados.append({
        **{k: params[k] for k in param_names},
        "RMSE_CV_medio": float(np.mean(fold_rmses)),
        "RMSE_CV_dp":    float(np.std(fold_rmses)),
    })
    if (j + 1) % 50 == 0 or (j + 1) == len(combos):
        elapsed = time.time() - t0
        print(f"  {j+1}/{len(combos)} combinações | {elapsed/60:.1f} min decorridos")

tempo_grid = time.time() - t0
print(f"\nBusca em grade concluída em {tempo_grid/60:.1f} minutos.")


In [ ]:
# Top-5 candidatos (menor RMSE médio de CV)
resultados_df = pd.DataFrame(resultados).sort_values("RMSE_CV_medio").reset_index(drop=True)
print("Top 5 combinações de hiperparâmetros:")
top5 = resultados_df.head(5)
print(top5.to_string(index=False))

best = resultados_df.iloc[0]
best_params = {k: best[k] for k in param_names}
# Coerce tipos inteiros
for k in ("n_estimators", "max_depth", "min_child_weight"):
    best_params[k] = int(best_params[k])

print("\nHiperparâmetros selecionados:")
for k, v in best_params.items():
    print(f"  {k:18s}: {v}")
print(f"\nRMSE CV médio: {best['RMSE_CV_medio']:.4f} p.p.")
print(f"Desvio padrão entre folds: {best['RMSE_CV_dp']:.4f} p.p.")


## 4.2.3 — Diagnóstico do modelo final

Treinamento final na janela completa de treino, comparação RMSE in-sample vs CV (overfitting),
curva de aprendizado (loss × número de árvores) e tempo de treinamento.


In [ ]:
params_final = {**base_params, **best_params, "eval_metric": "rmse"}

# Curva de aprendizado: usamos o último fold temporal como conjunto de validação,
# garantindo que a curva reflete dinâmica temporal genuína (não vazamento).
last_tr_idx, last_va_idx = list(tscv.split(X_treino))[-1]

t1 = time.time()
model_curva = xgb.XGBRegressor(**params_final)
model_curva.fit(
    X_treino.iloc[last_tr_idx], y_treino.iloc[last_tr_idx],
    eval_set=[
        (X_treino.iloc[last_tr_idx], y_treino.iloc[last_tr_idx]),
        (X_treino.iloc[last_va_idx], y_treino.iloc[last_va_idx]),
    ],
    verbose=False,
)
tempo_curva = time.time() - t1

evals = model_curva.evals_result()
rmse_train_curve = evals["validation_0"]["rmse"]
rmse_val_curve   = evals["validation_1"]["rmse"]
print(f"Modelo de curva treinado em {tempo_curva:.2f} s sobre o último fold.")


In [ ]:
# Modelo final sobre a janela completa de treino
t2 = time.time()
model_final = xgb.XGBRegressor(**params_final)
model_final.fit(X_treino, y_treino, verbose=False)
tempo_final = time.time() - t2

pred_in_sample = model_final.predict(X_treino)
rmse_in_sample = rmse(y_treino, pred_in_sample)

print(f"RMSE in-sample (treino completo):       {rmse_in_sample:.4f} p.p.")
print(f"RMSE médio na CV (out-of-sample):       {best['RMSE_CV_medio']:.4f} p.p.")
print(f"Gap (CV - in-sample):                   {best['RMSE_CV_medio'] - rmse_in_sample:+.4f} p.p.")
print(f"Tempo de treinamento do modelo final:   {tempo_final:.2f} s")
print(f"Tempo total (grid + curva + final):     {(tempo_grid + tempo_curva + tempo_final)/60:.1f} min")


In [ ]:
# Figura 8 — Curva de aprendizado do XGBoost
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(range(1, len(rmse_train_curve) + 1), rmse_train_curve,
        label="Treino", color="#1f4e79", linewidth=1.6)
ax.plot(range(1, len(rmse_val_curve) + 1), rmse_val_curve,
        label="Validação (último fold)", color="#c0504d", linewidth=1.6)
ax.set_xlabel("Número de árvores")
ax.set_ylabel("RMSE (p.p.)")
ax.set_title("Figura 8 — Curva de aprendizado do XGBoost")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig("figura_8_curva_aprendizado.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
# Tabela 3 — Hiperparâmetros selecionados
tabela_3 = pd.DataFrame({
    "Hiperparâmetro": param_names,
    "Valor selecionado": [best_params[k] for k in param_names],
    "Valores avaliados": [str(param_grid[k]) for k in param_names],
})
display(tabela_3)
tabela_3.to_csv("tabela_3_hiperparametros.csv", index=False)
folds_df.to_csv("tabela_folds.csv", index=False)
resultados_df.head(10).to_csv("tabela_top10_combos.csv", index=False)
print("Arquivos gerados: figura_8_curva_aprendizado.png, tabela_3_hiperparametros.csv, "
      "tabela_folds.csv, tabela_top10_combos.csv")


In [ ]:
# Persiste o modelo final (utilizado nas seções 4.3 e 4.4)
import joblib
joblib.dump({
    "model": model_final,
    "feature_cols": feature_cols,
    "best_params": best_params,
    "rmse_in_sample": rmse_in_sample,
    "rmse_cv_medio": float(best["RMSE_CV_medio"]),
    "rmse_cv_dp": float(best["RMSE_CV_dp"]),
    "tempos_segundos": {"grid": tempo_grid, "curva": tempo_curva, "final": tempo_final},
}, "modelo_xgb_final.joblib")
print("Modelo salvo em modelo_xgb_final.joblib")


In [ ]:
# Download dos artefatos
for nome in [
    "figura_8_curva_aprendizado.png",
    "tabela_3_hiperparametros.csv",
    "tabela_folds.csv",
    "tabela_top10_combos.csv",
    "modelo_xgb_final.joblib",
]:
    files.download(nome)
